# Lyric Engine — Kaggle Training Notebook (T4 x2)

**Repo:** https://github.com/SMXFREEZE/lyric-engine  
**Runtime:** GPU T4 x2 (32 GB VRAM total) — Settings → Accelerator → GPU T4 x2

### Stages
1. Check GPU (multi-GPU)
2. Install dependencies
3. Clone / update repo from GitHub
4. Config (tokens, model, genre) — **fp16, no quantization needed on 32 GB**
5. Scrape lyrics from Genius API
6. Build style vectors
7. **Viral DNA — scrape 55-country charts → compute trend signal**
8. Verify pipeline
9. Load model + LoRA (fp16, device_map=auto)
10. Train (with viral conditioning vector)
11. Test generation
12. **Generate full song (MusicGen large + Bark vocals)**
13. Save checkpoint

In [ ]:
# -- 1. Check GPU --
import torch

if not torch.cuda.is_available():
    print('NO GPU — go to Settings → Accelerator → GPU T4 x2')
    raise SystemExit('Enable GPU first')

n_gpus = torch.cuda.device_count()
total_vram = 0
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram
    print(f'GPU {i}: {name}  ({vram:.1f} GB)')

print(f'\nTotal VRAM : {total_vram:.1f} GB')
print(f'GPUs       : {n_gpus}')

if total_vram >= 28:
    print('✓ T4 x2 — will use fp16, no quantization needed')
elif total_vram >= 14:
    print('⚠ Single T4 — will use 4-bit quantization')
else:
    print('⚠ Low VRAM — switching to 4-bit + small model')

!nvidia-smi

In [ ]:
# -- 2. Install dependencies --
!pip install -q \
    transformers peft accelerate bitsandbytes trl \
    datasets lyricsgenius billboard.py \
    pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer python-dotenv \
    audiocraft suno-bark pydub soundfile
print('All dependencies installed.')

In [ ]:
# -- 3. Clone / update repo --
import os, sys

REPO = 'https://github.com/SMXFREEZE/lyric-engine'
DEST = '/kaggle/working/lyric-engine'

if os.path.exists(f'{DEST}/.git'):
    print('Already cloned — pulling latest...')
    !git -C {DEST} pull origin main
else:
    print('Cloning repo...')
    !git clone {REPO} {DEST}

%cd {DEST}
print('\nLatest commits:')
!git log --oneline -5

sys.path.insert(0, '.')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
print(f'\nCheckpoints → {CHECKPOINT_DIR}')

In [ ]:
# -- 4. Config --
# !! Add your tokens as Kaggle Secrets (eye icon on left sidebar) !!
# Secret name: GENIUS_TOKEN  → your Genius client access token
# Secret name: HF_TOKEN      → your HuggingFace token (write access)

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GENIUS_TOKEN = secrets.get_secret('GENIUS_TOKEN')
HF_TOKEN     = secrets.get_secret('HF_TOKEN')

os.environ['GENIUS_TOKEN']            = GENIUS_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# ── Auto-detect best settings based on available VRAM ────────────────────────
import torch
total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory / 1e9
    for i in range(torch.cuda.device_count())
)

# T4 x2 = 32 GB → fp16, no quantization, larger batches, bigger models
# Single T4   = 16 GB → 4-bit, smaller batches
USE_4BIT       = total_vram < 20          # False on T4 x2
MUSICGEN_SIZE  = 'large' if total_vram >= 20 else 'small'
LORA_RANK      = 64  if total_vram >= 20 else 16     # bigger adapter on more VRAM
BATCH_SIZE     = 4   if total_vram >= 20 else 2
GRAD_ACCUM     = 8   if total_vram >= 20 else 16     # eff batch = 32 either way

# ── Model ──────────────────────────────────────────────────────────────────────
# Mistral 7B fp16  ≈ 14 GB — fits comfortably in 32 GB leaving room for LoRA + activations
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'

# ── Training config ────────────────────────────────────────────────────────────
GENRE      = 'trap'   # trap / rnb / indie / pop / drill / hip_hop / country / latin / afrobeats / kpop
EPOCHS     = 3        # more epochs on T4 x2 since we have the power
LR         = 2e-4

# ── Scraper config ─────────────────────────────────────────────────────────────
MAX_SONGS_PER_ARTIST = 50   # more data, we have the time

# ── Chart config ───────────────────────────────────────────────────────────────
SCRAPE_CHARTS  = True   # set False to skip viral DNA if rate-limited
HF_DATASET_REPO = 'SMXFREEZE/lyric-engine-data'   # optional push target

print(f'Base model    : {BASE_MODEL}')
print(f'Genre         : {GENRE}')
print(f'Total VRAM    : {total_vram:.1f} GB')
print(f'4-bit quant   : {USE_4BIT}')
print(f'MusicGen size : {MUSICGEN_SIZE}')
print(f'LoRA rank     : {LORA_RANK}')
print(f'Eff. batch    : {BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs        : {EPOCHS}')
print(f'Genius token  : {"✓" if GENIUS_TOKEN else "✗ — add to Kaggle Secrets"}')
print(f'HF token      : {"✓" if HF_TOKEN else "✗ — add to Kaggle Secrets"}')

In [ ]:
# -- 5. Scrape lyrics — seeds + viral chart artists + discovery expansion --
#
# Strategy:
#   1. Start with genre seed artists
#   2. Pull charting artists from Billboard/Deezer/iTunes (populated in cell 7 if run first,
#      otherwise we do a quick chart pull here)
#   3. Discover new artists by following "featured artist" links on Genius
#   4. Push every song to HuggingFace dataset — persists across sessions
#   5. On next run: load existing HF dataset first so we never re-scrape
#
# Result: dataset grows every run, no cap on size

import json, time, re, os
import lyricsgenius
from collections import deque
from tqdm.auto import tqdm

# ── Genre seed artists (starting points) ─────────────────────────────────────
GENRE_SEEDS = {
    'trap':      ['Future', 'Young Thug', 'Gunna', 'Lil Baby', '21 Savage', 'Roddy Ricch', 'Offset', 'Playboi Carti', 'Travis Scott', 'Metro Boomin'],
    'rnb':       ['Frank Ocean', 'SZA', 'H.E.R.', 'Bryson Tiller', 'Summer Walker', 'The Weeknd', 'Brent Faiyaz', 'Giveon', 'Lucky Daye'],
    'indie':     ['Phoebe Bridgers', 'Sufjan Stevens', 'Bon Iver', 'Mitski', 'Clairo', 'Soccer Mommy', 'Big Thief', 'Snail Mail'],
    'pop':       ['Taylor Swift', 'Olivia Rodrigo', 'Dua Lipa', 'Ariana Grande', 'Sabrina Carpenter', 'Charli XCX', 'Billie Eilish', 'Harry Styles'],
    'drill':     ['Pop Smoke', 'Fivio Foreign', 'Lil Durk', 'King Von', 'Central Cee', 'Digga D', 'Headie One', 'Unknown T'],
    'alt_emo':   ['Paramore', 'My Chemical Romance', 'Lana Del Rey', 'Conan Gray', 'Gracie Abrams', 'Olivia O'Brien'],
    'hip_hop':   ['Kendrick Lamar', 'J. Cole', 'Drake', 'Jay-Z', 'Nas', 'Lupe Fiasco', 'Pusha T', 'Joey Bada$$'],
    'country':   ['Morgan Wallen', 'Luke Combs', 'Zach Bryan', 'Kacey Musgraves', 'Tyler Childers', 'Chris Stapleton'],
    'latin':     ['Bad Bunny', 'J Balvin', 'Karol G', 'Peso Pluma', 'Rauw Alejandro', 'Feid', 'Maluma', 'Ozuna'],
    'afrobeats': ['Burna Boy', 'Wizkid', 'Davido', 'Rema', 'Tems', 'Asake', 'Ayra Starr', 'Omah Lay'],
    'french_rap':['PNL', 'Hamza', 'Ninho', 'SCH', 'Niska', 'Freeze Corleone', 'Gazo'],
    'kpop':      ['BTS', 'BLACKPINK', 'Stray Kids', 'aespa', 'NewJeans', 'TWICE', 'EXO'],
    'arabic_pop':['Amr Diab', 'Cairokee', 'Marwan Pablo', 'El Waili', 'Wegz', 'Afroto'],
    'chaabi':    ['Cheb Khaled', 'Cheb Mami', 'Rachid Taha', 'Faudel'],
    'rnb':       ['Frank Ocean', 'SZA', 'The Weeknd', 'Brent Faiyaz', 'Summer Walker'],
}

# ── Pull viral chart artist names (from cell 7 if available) ─────────────────
chart_artists = set()
chart_data_path = 'data/charts'
if os.path.exists(chart_data_path):
    import pathlib
    for f in pathlib.Path(chart_data_path).glob('*.json'):
        try:
            data = json.loads(f.read_text(encoding='utf-8'))
            for entry in data if isinstance(data, list) else data.values():
                if isinstance(entry, dict) and 'artist' in entry:
                    chart_artists.add(entry['artist'])
                elif isinstance(entry, list):
                    for item in entry:
                        if isinstance(item, dict) and 'artist' in item:
                            chart_artists.add(item['artist'])
        except Exception:
            pass
    print(f'Found {len(chart_artists)} artists from viral charts')

# Combine: genre seeds + viral chart artists
seed_artists = list(set(GENRE_SEEDS.get(GENRE, GENRE_SEEDS['trap'])) | chart_artists)
print(f'Total starting artists: {len(seed_artists)} (seeds + viral chart artists)')

# ── Load existing HF dataset to avoid re-scraping ────────────────────────────
existing_titles = set()
out_path = f'data/raw/{GENRE}.jsonl'
os.makedirs('data/raw', exist_ok=True)

try:
    from datasets import load_dataset
    print(f'\nLoading existing dataset from HuggingFace: {HF_DATASET_REPO}...')
    existing_ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
    # Filter to this genre
    genre_rows = [r for r in existing_ds if r.get('genre') == GENRE]
    existing_titles = {r['title'] for r in genre_rows}
    # Save locally
    with open(out_path, 'w', encoding='utf-8') as f:
        for r in genre_rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Loaded {len(genre_rows)} existing {GENRE} songs from HF (skip re-scraping these)')
except Exception as e:
    print(f'HF load skipped ({e}) — starting fresh')

# ── Scrape function ───────────────────────────────────────────────────────────
def clean_lyrics(raw):
    text = re.sub(r'\[.*?\]', '', raw)
    text = re.sub(r'\d+ Contributors?.*?Lyrics', '', text, flags=re.DOTALL)
    text = re.sub(r'\d*Embed$', '', text.strip())
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    verbose=False,
    remove_section_headers=True,
    skip_non_songs=True,
    timeout=15,
)

songs = []          # new songs this run
discovered = set()  # artists we've already processed
queue = deque(seed_artists)
NEW_ARTIST_LIMIT = 60     # expand up to 60 artists via featured-artist discovery

print(f'\nStarting scrape — will expand to ~{NEW_ARTIST_LIMIT} artists via discovery...')

with open(out_path, 'a', encoding='utf-8') as out_f:
    pbar = tqdm(total=NEW_ARTIST_LIMIT, desc='Artists scraped')
    while queue and len(discovered) < NEW_ARTIST_LIMIT:
        artist_name = queue.popleft()
        if artist_name in discovered:
            continue
        discovered.add(artist_name)
        pbar.update(1)
        pbar.set_description(f'Scraping: {artist_name[:30]}')

        try:
            artist = genius.search_artist(artist_name, max_songs=MAX_SONGS_PER_ARTIST, sort='popularity')
            if not artist:
                continue

            new_this_artist = 0
            for song in artist.songs:
                # Skip if already in HF dataset
                if song.title in existing_titles:
                    continue
                if not song.lyrics or len(song.lyrics.split()) < 80:
                    continue
                lyrics = clean_lyrics(song.lyrics)
                if len(lyrics.split()) < 80:
                    continue

                row = {
                    'artist': artist_name,
                    'title':  song.title,
                    'genre':  GENRE,
                    'lyrics': lyrics,
                }
                out_f.write(json.dumps(row, ensure_ascii=False) + '\n')
                songs.append(row)
                existing_titles.add(song.title)
                new_this_artist += 1

                # ── Artist discovery: queue featured artists ──────────────────
                try:
                    for feat in (song.featured_artists or []):
                        fname = feat.get('name', '') if isinstance(feat, dict) else str(feat)
                        if fname and fname not in discovered and fname not in queue:
                            queue.append(fname)
                except Exception:
                    pass

            time.sleep(1.0)
        except Exception as e:
            pbar.write(f'  Error: {artist_name}: {e}')
            time.sleep(2.0)

    pbar.close()

# Count total (existing + new)
total_songs = len(existing_titles)
print(f'\nNew songs scraped this run : {len(songs)}')
print(f'Total in dataset           : {total_songs}')
print(f'Artists processed          : {len(discovered)}')
print(f'Artists still in queue     : {len(queue)}')

# ── Push to HuggingFace ───────────────────────────────────────────────────────
if len(songs) > 0 and HF_TOKEN:
    try:
        from datasets import load_dataset, Dataset
        import pandas as pd

        print(f'\nPushing {len(songs)} new songs to HuggingFace: {HF_DATASET_REPO}...')

        # Load full existing dataset and append
        try:
            existing_ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
            df_existing = existing_ds.to_pandas()
        except Exception:
            df_existing = pd.DataFrame()

        df_new = pd.DataFrame(songs)
        df_full = pd.concat([df_existing, df_new], ignore_index=True).drop_duplicates(subset=['artist', 'title'])

        Dataset.from_pandas(df_full).push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)
        print(f'✓ Pushed — HF dataset now has {len(df_full)} songs total')
    except Exception as e:
        print(f'HF push failed: {e} — songs saved locally at {out_path}')
else:
    print('No new songs to push or HF token missing.')

In [ ]:
# -- 6. Build style vectors --
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=2,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# -- 7. Viral DNA — scrape 55-country charts + compute trend conditioning vector --
import json
import numpy as np

VIRAL_VECTOR_PATH = 'data/viral_conditioning.npy'
VIRAL_REPORT_PATH = 'data/viral_report.json'

if SCRAPE_CHARTS:
    print('Scraping charts from Billboard / Deezer / Spotify CSV / iTunes...')
    print('(Covers 55+ countries including Morocco, all of Africa, Asia, LatAm, Europe)\n')

    try:
        from src.data.chart_scraper import run_weekly, compute_viral_scores
        from src.data.viral_analyzer import run_analysis, build_conditioning_vector

        # 1. Fetch charts (this hits free public APIs — no login needed)
        chart_data = run_weekly(output_dir='data/charts')
        print(f'  Charts fetched: {len(chart_data)} country/source entries')

        # 2. Compute viral scores (songs charting in many countries = viral DNA)
        viral_scores = compute_viral_scores(chart_data)
        top_viral = sorted(viral_scores.items(), key=lambda x: x[1], reverse=True)[:20]
        print('\nTop 20 globally viral tracks:')
        for title, score in top_viral:
            print(f'  [{score:5.1f}] {title}')

        # 3. Run full viral analysis (region profiles, trend velocity, universal DNA)
        report = run_analysis(chart_data, output_dir='data/viral_analysis')

        # 4. Build 32-dim conditioning vector for model training
        viral_vec = build_conditioning_vector(report)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

        with open(VIRAL_REPORT_PATH, 'w') as f:
            json.dump(report, f, indent=2, default=str)

        print(f'\nViral conditioning vector shape : {viral_vec.shape}')
        print(f'Saved to: {VIRAL_VECTOR_PATH}')
        print('\nRegion DNA profiles:')
        for region, profile in report.get('region_profiles', {}).items():
            print(f'  {region:15s}: BPM={profile.get("avg_bpm", "?"):.0f}  energy={profile.get("avg_energy", "?"):.2f}')

    except Exception as e:
        print(f'Chart scraping error: {e}')
        print('Falling back to neutral conditioning vector...')
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

else:
    print('SCRAPE_CHARTS=False — loading saved vector or using neutral...')
    if os.path.exists(VIRAL_VECTOR_PATH):
        viral_vec = np.load(VIRAL_VECTOR_PATH)
        print(f'Loaded: {VIRAL_VECTOR_PATH}')
    else:
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)
        print('Using neutral zero vector')

print(f'\nViral conditioning vector (first 8 dims): {viral_vec[:8].round(3)}')
print('Viral DNA ready ✓')

In [ ]:
# -- 7. Verify pipeline --
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
print(f"Artist : {sample['artist']}")
print(f"Title  : {sample['title']}")
print()

scheme = detect_scheme(lines)
print(f"Rhyme scheme  : {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density : {scheme['rhyme_density']}")
print()

for em in score_lyrics('\n'.join(lines)):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables:2d}syl] {em.text[:60]}")

print('\nPipeline OK ✓')

In [ ]:
# -- 9. Load base model + LoRA (fp16 on T4 x2, 4-bit fallback on single GPU) --
import torch
from src.model.lyrics_model import load_base_model, apply_lora, LyricsModel

print(f'Loading {BASE_MODEL}...')
print(f'Mode: {"4-bit quantization" if USE_4BIT else "fp16 full precision"}')
print(f'device_map=auto → will spread across {torch.cuda.device_count()} GPU(s)\n')

base, tokenizer = load_base_model(
    BASE_MODEL,
    use_4bit=USE_4BIT,
    device=None,        # let device_map='auto' handle distribution
)

# Apply LoRA adapter
base = apply_lora(base, rank=LORA_RANK, alpha=LORA_RANK * 2)

# Wrap with phonetic head
model = LyricsModel(base, d_model=base.config.hidden_size)

base.print_trainable_parameters()

# Show which layers are on which GPU
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f}/{total:.1f} GB used')

print('\nModel ready ✓')

In [ ]:
# -- 10. Train genre LoRA (with viral DNA conditioning) --
import numpy as np
from src.training.sft import train_sft

# Load the viral conditioning vector built in cell 7
viral_vec = np.load(VIRAL_VECTOR_PATH) if os.path.exists(VIRAL_VECTOR_PATH) else np.zeros(32, dtype=np.float32)
print(f'Viral conditioning vector loaded: {viral_vec.shape}, norm={np.linalg.norm(viral_vec):.3f}')

# Stage 1: General foundation (high rank, all data) — skipped if checkpoint exists
stage1_ckpt = f'{CHECKPOINT_DIR}/stage1_general'
if not os.path.exists(stage1_ckpt):
    print('\n[Stage 1] Training general foundation LoRA (rank 64)...')
    train_sft(
        stage=1,
        genre=GENRE,
        data_path=f'data/raw/{GENRE}.jsonl',
        base_model=BASE_MODEL,
        output_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM,
        epochs=1,
        lr=LR,
        lora_rank=64,
        alpha=128,
        use_4bit=USE_4BIT,
        style_vec_dir='data/style_vectors',
        viral_conditioning=viral_vec,
    )
    print('Stage 1 done ✓')
else:
    print(f'Stage 1 checkpoint found at {stage1_ckpt} — skipping')

# Stage 2: Genre-specific LoRA (smaller rank, fine-tuned per genre)
print(f'\n[Stage 2] Training {GENRE} genre LoRA (rank {LORA_RANK})...')
train_sft(
    stage=2,
    genre=GENRE,
    data_path=f'data/raw/{GENRE}.jsonl',
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=EPOCHS,
    lr=LR,
    lora_rank=LORA_RANK,
    alpha=LORA_RANK * 2,
    use_4bit=USE_4BIT,
    style_vec_dir='data/style_vectors',
    viral_conditioning=viral_vec,   # ← viral DNA wired in as training signal
)
print(f'\nTraining complete ✓ — checkpoint: {CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}')

In [ ]:
# -- 10. Test generation --
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'

print(f'Loading checkpoint from {ckpt_path}...')
tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, load_in_4bit=True, device_map='auto'
)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f'\n=== Generated {GENRE.upper()} verse ===')
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for i, line in enumerate(verse, 1):
    print(f'  {i}. {line}')

In [ ]:
# -- 12. Generate full song (MusicGen large + Bark vocals + final MP3) --
# Uses the lyrics generated in cell 11 above + the trained genre style

from src.audio.song_assembler import SongAssembler, SongSpec

# Grab generated verse from test cell above (or define custom)
# 'verse' is the list of lines generated in cell 11
generated_lyrics = {
    'verse1':  verse[:4] if len(verse) >= 4 else verse,
    'chorus':  verse[4:8] if len(verse) >= 8 else [
        'Turn up the heat, we burning bright tonight',
        'Everything I touch turns gold in the light',
    ],
    'verse2':  verse[:4] if len(verse) >= 4 else verse,
    'chorus2': verse[4:8] if len(verse) >= 8 else [
        'Turn up the heat, we burning bright tonight',
        'Everything I touch turns gold in the light',
    ],
}

spec = SongSpec(
    title=f'AI_{GENRE.upper()}_001',
    genre=GENRE,
    bpm=140 if GENRE in ('trap', 'drill') else 120,
    mood='dark' if GENRE in ('trap', 'drill', 'alt_emo') else 'hype',
    language='en',
    voice_idx=0,
    lyrics=generated_lyrics,
    sections=[
        ('intro',   8),
        ('verse1',  16),
        ('chorus',  12),
        ('verse2',  16),
        ('chorus2', 12),
        ('bridge',  8),
        ('outro',   8),
    ],
    output_dir='/kaggle/working/songs',
)

print(f'Assembling song: {spec.title}')
print(f'MusicGen size : {MUSICGEN_SIZE}  (large = best quality, uses ~6 GB VRAM)')
print(f'Bark          : small model (vocal synthesis, ~2 GB VRAM)')
print(f'Total sections: {[s[0] for s in spec.sections]}')
print()

# Free LLM VRAM before loading audio models
import gc
try:
    del base, model
    gc.collect()
    torch.cuda.empty_cache()
    print('LLM unloaded to free VRAM for audio models')
except Exception:
    pass

# Show VRAM state before audio generation
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f}/{total:.1f} GB free after LLM unload')

assembler = SongAssembler(
    musicgen_size=MUSICGEN_SIZE,
    bark_small=True,
    device='auto',
)

song = assembler.generate_song(spec, skip_vocals=False, skip_instrumental=False)

print(f'\n{"="*60}')
print(f'Song generated successfully!')
print(f'Output: {song.output_path}')
print(f'Duration: {len(song.mixed) / 44100:.1f}s')
print(f'\nFiles in /kaggle/working/songs/:')
import os, pathlib
for p in sorted(pathlib.Path('/kaggle/working/songs').rglob('*')):
    if p.is_file():
        size = p.stat().st_size / 1e6
        print(f'  {p.name:50s} {size:.1f} MB')
print('\nDownload from: Kaggle → Output tab (right sidebar) → songs/')

In [ ]:
# -- 11. Checkpoint summary --
import os

ckpt = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'
print(f'Checkpoint: {ckpt}')
print()
if os.path.exists(ckpt):
    total_mb = 0
    for fname in sorted(os.listdir(ckpt)):
        size = os.path.getsize(f'{ckpt}/{fname}')
        total_mb += size / 1e6
        print(f'  {fname:45s} {size/1e6:6.1f} MB')
    print(f'\n  Total: {total_mb:.0f} MB')
else:
    print('Checkpoint directory not found — did training complete?')

print()
print('To download: Kaggle → Output tab (right sidebar) → Download')